In [13]:
import numpy as np
import scipy.io as sio
import os
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.spatial.distance import cdist
import matplotlib.tri as mtri
import scipy.spatial as spatial
import triangle as tr
from scipy.integrate import dblquad

# 设置中文字体
plt.rcParams["font.family"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

def generate_spherical_mesh(n_nodes=500):
    """在单位球面上生成三角形网格"""
    indices = np.arange(0, n_nodes, dtype=float) 
    phi = np.arccos(1 - 2 * indices / n_nodes)  # 极角
    theta = np.pi * (1 + 5**0.5) * indices  # 方位角
    
    # 球面坐标转笛卡尔坐标
    x = np.cos(theta) * np.sin(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(phi)
    
    # 立体投影到赤道平面(z=0)
    k = 1 / (1 + z)
    u = x * k
    v = y * k
    
    # 平面三角剖分
    points = np.column_stack((u, v))
    tri = tr.triangulate({'vertices': points}, 'e')
    
    # 提取平面坐标并映射回球面
    vertices = tri['vertices']
    triangles = tri['triangles']
    u = vertices[:, 0]
    v = vertices[:, 1]
    d = u**2 + v**2
    x = 2 * u / (1 + d)
    y = 2 * v / (1 + d)
    z = (1 - d) / (1 + d)
    
    # 单位球归一化（数值稳定性）
    norms = np.sqrt(x**2 + y**2 + z**2)
    x /= norms
    y /= norms
    z /= norms
    
    nodes = np.column_stack((x, y, z))
    return nodes, triangles


def v1(x, y, z):
    """球面上的正交函数1：基于球谐函数Y₁₀，调整系数使y值在100左右"""
    # 经过计算，这个缩放因子能使最终y值在100左右
    scale_factor = 100 # 关键调整：通过计算得到的缩放因子
    coefficient = scale_factor * np.sqrt(3 / (4 * np.pi))
    return coefficient * z 

def v2(x, y, z):
    """球面上的正交函数2：基于球谐函数Y₁₁，调整系数使y值在100左右"""
    # 保持与v1相同的缩放因子，确保两个正交函数贡献相当
    scale_factor = 100 # 关键调整：与v1相同的缩放因子
    coefficient = -scale_factor * np.sqrt(3 / (8 * np.pi))
    return coefficient * x 

def generate_p_matrices(nodes, n_samples=1):
    """生成对称P矩阵列表：1.上三角U(0,1)采样；2.对称填充；3.元素和=1"""
    P_list = []
    n_nodes = len(nodes)

    for i in range(n_samples):
        triu_idx = np.triu_indices(n_nodes)
        P_upper = np.zeros((n_nodes, n_nodes), dtype=np.float64)
        P_upper[triu_idx] = np.random.uniform(0, 1, size=len(triu_idx[0])).astype(np.float64)
        
        P_sym = P_upper + P_upper.T - np.diag(P_upper.diagonal())
        
        sum_P = P_sym.sum()
        P_norm = P_sym / sum_P
        
        P_scaled = P_norm * 1
        
        rank_P = np.linalg.matrix_rank(P_scaled)
        if (i + 1) % 10 == 0:
            print(f"  已生成 {i+1}/{n_samples} 个P矩阵：")
            print(f"    - 维度: {P_scaled.shape}, 秩: {rank_P}, 元素和: {P_scaled.sum():.6f}")
        
        P_list.append(P_scaled)

    return P_list

def generate_b_vector(nodes):
    """生成指定的两个b向量"""
    bs = []
    x, y, z = nodes[:, 0], nodes[:, 1], nodes[:, 2]
    
    b1 = (v1(x, y, z) * 1).astype(np.float64)
    b2 = (v2(x, y, z) * 1).astype(np.float64)
    
    bs.append(b1)
    bs.append(b2)
    return bs

def generate_response_y(P_list, bs, lambda_vals, mu_true=1):
    """生成响应变量y，噪声标准差为条件期望部分标准差的10%"""
    n_samples = len(P_list)
    K = len(bs)
    
    # 计算条件期望（无噪声部分）
    y_noiseless = np.full(n_samples, mu_true, dtype=np.float64)
    for i in range(n_samples):
        for k in range(K):
            quadratic_form = bs[k].T @ P_list[i] @ bs[k]
            y_noiseless[i] += lambda_vals[k] * quadratic_form
    
    # 计算条件期望的标准差
    epsilon_std = 0.1 * np.std(y_noiseless)
    print(f"\n条件期望标准差: {np.std(y_noiseless):.6e}")
    print(f"噪声标准差 (10%): {epsilon_std:.6e}")
    
    # 生成噪声并添加到响应变量
    noise = np.random.normal(0, epsilon_std, n_samples).astype(np.float64)
    y = y_noiseless + noise
    
    # 计算信噪比
    signal_range = np.max(y_noiseless) - np.min(y_noiseless)
    snr = signal_range / epsilon_std if epsilon_std != 0 else np.inf
    print(f"信号统计: 均值 = {np.mean(y_noiseless):.6e}, 范围 = {signal_range:.6e}, SNR = {snr:.2f}")
    return y

def save_precision_data(file_path, data_dict):
    """高精度保存数据，确保与MATLAB兼容"""
    try:
        for key, val in data_dict.items():
            if isinstance(val, np.ndarray):
                data_dict[key] = val.astype(np.float64)
        sio.savemat(file_path, data_dict, do_compression=False)
        return True
    except Exception as e:
        print(f"保存数据时出错: {str(e)}")
        return False

# 主程序
if __name__ == "__main__":
    np.random.seed(123)  

    print("1. 生成球面网格...")
    nodes, triangles = generate_spherical_mesh(n_nodes=500)
    n_nodes = len(nodes)
    print(f"   网格生成完成: {n_nodes}个节点, {len(triangles)}个三角形")

    print("\n2. 设置真实参数...")
    K = 2  
    lambda_vals = [1, 1]  # 保持lambda值不变
    mu_true = 100

    print("\n3. 生成指定的b向量（v1和v2）...")
    bs = generate_b_vector(nodes)
    for k in range(K):
        b = bs[k]
        non_zero_count = np.count_nonzero(b)
        print(f"\nb向量 {k+1} 统计:")
        print(f"  非零元素数量: {non_zero_count}")
        print(f"  值范围: [{np.min(b):.6f}, {np.max(b):.6f}]")
        print(f"  均值: {np.mean(b):.6e}, 标准差: {np.std(b):.6e}")
    print(f"\n   真实参数：lambda_vals = {lambda_vals}, mu_true = {mu_true}")

    print("\n4. 生成P矩阵（对称、秩>1、元素和=1）...")
    P_list = generate_p_matrices(nodes, n_samples=50)  
    print(f"   P矩阵生成完成: {len(P_list)}个矩阵, 每个维度 {P_list[0].shape}")

    print("\n5. 生成响应变量y...")
    y = generate_response_y(P_list, bs, lambda_vals, mu_true)

    print("\n   y值统计:")
    print(f"   均值 = {np.mean(y):.10e}, 范围 = [{np.min(y):.10e}, {np.max(y):.10e}]")
    print(f"   标准差 = {np.std(y):.10e}")
    print(f"   前5个y: {np.round(y[:50], 10)}")

    print("\n6. 保存数据...")
    output_dir = 'simulation_data'
    os.makedirs(output_dir, exist_ok=True)

    y_saved = save_precision_data(
        f'{output_dir}/y.mat', 
        {'y': y}
    )

    for i, P_i in enumerate(P_list):
        p_saved = save_precision_data(
            f'{output_dir}/slice_{i+1}.mat', 
            {f'P_{i+1}': P_i}
        )

    true_params = {
        'mu_true': mu_true,
        'nodes': nodes,
        'triangles': triangles + 1,  # MATLAB索引从1开始
        'b_true': np.array(bs),
        'lambda_true': np.array(lambda_vals),
    }
    params_saved = save_precision_data(
        f'{output_dir}/true_params.mat', 
        true_params
    )

    if y_saved and params_saved:
        print(f"   数据已高精度保存到 {output_dir} 目录")
        print("   在MATLAB中查看时，使用`format long`命令可显示完整小数")
    else:
        print("   数据保存过程中出现错误")
        
    print("\n模拟数据生成过程结束！")


1. 生成球面网格...
   网格生成完成: 500个节点, 993个三角形

2. 设置真实参数...

3. 生成指定的b向量（v1和v2）...

b向量 1 统计:
  非零元素数量: 499
  值范围: [-48.664810, 48.860251]
  均值: 9.772050e-02, 标准差: 2.820942e+01

b向量 2 统计:
  非零元素数量: 499
  值范围: [-34.466938, 34.500186]
  均值: 5.066331e-03, 标准差: 1.994681e+01

   真实参数：lambda_vals = [1, 1], mu_true = 100

4. 生成P矩阵（对称、秩>1、元素和=1）...
  已生成 10/50 个P矩阵：
    - 维度: (500, 500), 秩: 500, 元素和: 1.000000
  已生成 20/50 个P矩阵：
    - 维度: (500, 500), 秩: 500, 元素和: 1.000000
  已生成 30/50 个P矩阵：
    - 维度: (500, 500), 秩: 500, 元素和: 1.000000
  已生成 40/50 个P矩阵：
    - 维度: (500, 500), 秩: 500, 元素和: 1.000000
  已生成 50/50 个P矩阵：
    - 维度: (500, 500), 秩: 500, 元素和: 1.000000
   P矩阵生成完成: 50个矩阵, 每个维度 (500, 500)

5. 生成响应变量y...

条件期望标准差: 1.601594e+00
噪声标准差 (10%): 1.601594e-01
信号统计: 均值 = 1.003917e+02, 范围 = 5.883253e+00, SNR = 36.73

   y值统计:
   均值 = 1.0036444854e+02, 范围 = [9.7093443761e+01, 1.0350960336e+02]
   标准差 = 1.6553921352e+00
   前5个y: [101.44367623  99.63779284 101.03415808  98.89811945  98.2631618
  99.61333095  99.86